# Grafana y Prometheus para la telemetría de Vault

Este notebook instala `kube-prometheus-stack`, configura un scrape autenticado de `/v1/sys/metrics` y carga un dashboard de Vault en Grafana. El token de métricas tiene una política mínima y solo se almacena en un Secret de Kubernetes.

> Requisito: Vault debe tener `prometheus_retention_time` mayor que cero en su stanza `telemetry`. El notebook valida el endpoint antes del despliegue y no modifica el StatefulSet de Vault.

## 1. Variables y prerrequisitos

In [ ]:
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((d / ".env" for d in (Path.cwd(), *Path.cwd().parents) if (d / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("No se encontró el fichero .env")
load_dotenv(ENV_FILE, override=True)

required = ("VAULT_ADDR", "VAULT_TOKEN")
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Variables requeridas ausentes: {', '.join(missing)}")

settings = {
    "KUBE_CONTEXT": os.getenv("KUBE_CONTEXT", ""),
    "MONITORING_NAMESPACE": "vault-monitoring",
    "MONITORING_RELEASE": "vault-monitoring",
    "PROMETHEUS_POLICY": "vault-prometheus-metrics",
    "PROMETHEUS_TOKEN_TTL": "24h",
    "PROMETHEUS_STACK_VERSION": "82.10.1",
    "WORKDIR": "/tmp/vault-monitoring-demo",
    "AWS_REGION": "eu-central-1",
    "EKS_CLUSTER_NAME": "eks-infra-dev",
}
os.environ.update(settings)
Path(settings["WORKDIR"]).mkdir(parents=True, exist_ok=True)

subprocess.run(["doormat", "login", "-f"], check=True)
aws_env = subprocess.run(
    ["bash", "-lc", 'eval "$(doormat aws -a aws_jose.merchan_test export)" && env -0'],
    check=True,
    capture_output=True,
).stdout
for entry in aws_env.split(b"\0"):
    if entry.startswith(b"AWS_") and b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

subprocess.run([
    "aws", "eks", "update-kubeconfig",
    "--region", settings["AWS_REGION"],
    "--name", settings["EKS_CLUSTER_NAME"],
], check=True)

print(f"EKS: {settings['EKS_CLUSTER_NAME']} ({settings['AWS_REGION']})")
print(f"Vault: {os.environ['VAULT_ADDR']}")
print(f"Namespace Kubernetes: {settings['MONITORING_NAMESPACE']}")
print(f"Contexto: {settings['KUBE_CONTEXT'] or 'contexto actual'}")

In [ ]:
%%bash
set -euo pipefail

command -v vault >/dev/null
command -v kubectl >/dev/null
command -v helm >/dev/null
command -v jq >/dev/null
vault status >/dev/null
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} cluster-info >/dev/null
echo 'OK: Vault, Kubernetes, Helm y jq disponibles.'

## 2. Token de scrape con privilegio mínimo

La política permite únicamente leer métricas. El token no se imprime ni se escribe en disco: se canaliza directamente a un Secret de Kubernetes.

In [ ]:
%%bash
set -euo pipefail

vault policy write "${PROMETHEUS_POLICY}" - <<'EOF'
path "sys/metrics" {
  capabilities = ["read"]
}
EOF

kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} create namespace "${MONITORING_NAMESPACE}" --dry-run=client -o yaml | \
  kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} apply -f -

metrics_token=$(vault token create -policy="${PROMETHEUS_POLICY}" -period="${PROMETHEUS_TOKEN_TTL}" -orphan -format=json | jq -r '.auth.client_token')
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${MONITORING_NAMESPACE}" create secret generic vault-prometheus-token \
  --from-literal=token="${metrics_token}" --dry-run=client -o yaml | \
  kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} apply -f -

status=$(curl --silent --show-error --output /tmp/vault-metrics-sample.txt --write-out '%{http_code}' \
  --header "X-Vault-Token: ${metrics_token}" \
  --header 'Accept: application/openmetrics-text' \
  "${VAULT_ADDR}/v1/sys/metrics?format=prometheus")
unset metrics_token
test "${status}" = 200
grep -q '^vault_' /tmp/vault-metrics-sample.txt
rm -f /tmp/vault-metrics-sample.txt
echo 'OK: endpoint Prometheus de Vault accesible con el token restringido.'

## 3. Prometheus y Grafana

Prometheus obtiene el token desde el Secret y realiza el scrape cada 15 segundos. Grafana recibe un dashboard provisionado por su sidecar.

In [ ]:
%%bash
set -euo pipefail

cat > "${WORKDIR}/values.yaml" <<EOF
grafana:
  adminPassword: vault-demo
  service:
    type: LoadBalancer
    annotations:
      service.beta.kubernetes.io/aws-load-balancer-type: nlb
      service.beta.kubernetes.io/aws-load-balancer-scheme: internet-facing
  sidecar:
    dashboards:
      enabled: true
      label: grafana_dashboard
      labelValue: "1"
prometheus:
  prometheusSpec:
    secrets:
      - vault-prometheus-token
    additionalScrapeConfigs:
      - job_name: vault
        metrics_path: /v1/sys/metrics
        params:
          format: [prometheus]
        scheme: $(printf '%s' "${VAULT_ADDR}" | cut -d: -f1)
        bearer_token_file: /etc/prometheus/secrets/vault-prometheus-token/token
        tls_config:
          insecure_skip_verify: true
        static_configs:
          - targets:
              - $(printf '%s' "${VAULT_ADDR}" | sed -E 's#https?://##; s#/.*##')
        scrape_interval: 15s
        scrape_timeout: 10s
alertmanager:
  enabled: true
EOF

helm repo add prometheus-community https://prometheus-community.github.io/helm-charts --force-update
helm repo update
helm upgrade --install "${MONITORING_RELEASE}" prometheus-community/kube-prometheus-stack \
  --kube-context "${KUBE_CONTEXT:-$(kubectl config current-context)}" \
  --namespace "${MONITORING_NAMESPACE}" \
  --version "${PROMETHEUS_STACK_VERSION}" \
  --values "${WORKDIR}/values.yaml" \
  --wait --timeout 10m


In [ ]:
%%bash
set -euo pipefail

# Dashboard 12904 revision 2, normalized in manifest/ for direct provisioning:
# datasource Prometheus and the native Grafana piechart panel.
jq -e '.gnetId == 12904 and .uid == "vaults" and .title == "Hashicorp Vault"' \
  manifest/12904_rev2.json >/dev/null
if jq -e '.. | strings | select(contains("${DS_PROMXY}"))' manifest/12904_rev2.json >/dev/null; then
  echo 'El dashboard todavía contiene el placeholder DS_PROMXY.' >&2
  exit 1
fi
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${MONITORING_NAMESPACE}" create configmap vault-grafana-dashboard \
  --from-file=vault-overview.json="manifest/12904_rev2.json" \
  --dry-run=client -o yaml | \
  kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} label --local -f - grafana_dashboard=1 -o yaml | \
  kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} apply -f -
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${MONITORING_NAMESPACE}" rollout status deployment/"${MONITORING_RELEASE}"-grafana --timeout=5m
echo 'OK: dashboard Vault Overview provisionado.'

## 4. Validación y acceso

In [ ]:
%%bash
set -euo pipefail

prometheus_pod=$(kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${MONITORING_NAMESPACE}" get pod \
  -l app.kubernetes.io/name=prometheus -o jsonpath='{.items[0].metadata.name}')
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${MONITORING_NAMESPACE}" port-forward "${prometheus_pod}" 19090:9090 >/tmp/vault-prometheus-port-forward.log 2>&1 &
pf_pid=$!
trap 'kill ${pf_pid} 2>/dev/null || true' EXIT
for attempt in $(seq 1 30); do curl -fsS http://127.0.0.1:19090/-/ready >/dev/null && break; sleep 1; done
result=$(curl -fsS --get --data-urlencode 'query=up{job="vault"}' http://127.0.0.1:19090/api/v1/query)
jq . <<<"${result}"
jq -e '.data.result | length > 0 and all(.[]; .value[1] == "1")' <<<"${result}" >/dev/null
echo 'OK: Prometheus está recolectando telemetría de Vault.'

grafana_service="${MONITORING_RELEASE}-grafana"
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${MONITORING_NAMESPACE}" \
  port-forward "svc/${grafana_service}" 13000:80 >/tmp/vault-grafana-port-forward.log 2>&1 &
grafana_pf_pid=$!
trap 'kill ${pf_pid} ${grafana_pf_pid} 2>/dev/null || true' EXIT
for attempt in $(seq 1 60); do
  curl -fsS -u admin:vault-demo http://127.0.0.1:13000/api/health >/dev/null && break
  [[ "$attempt" -eq 60 ]] && { cat /tmp/vault-grafana-port-forward.log >&2; exit 1; }
  sleep 1
done
dashboard=$(curl -fsS -u admin:vault-demo \
  'http://127.0.0.1:13000/api/dashboards/uid/vaults')
jq -e '.dashboard.uid == "vaults" and .dashboard.title == "Hashicorp Vault"' \
  <<<"${dashboard}" >/dev/null

grafana_host=''
for attempt in $(seq 1 80); do
  grafana_host=$(kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} \
    -n "${MONITORING_NAMESPACE}" get service "${grafana_service}" \
    -o jsonpath='{.status.loadBalancer.ingress[0].hostname}')
  [[ -n "${grafana_host}" ]] && break
  sleep 5
done
: "${grafana_host:?Grafana NLB hostname no disponible}"
echo "OK: dashboard Hashicorp Vault (12904, UID vaults) cargado."
echo "Grafana: http://${grafana_host}"
echo 'Usuario: admin | Password de demo: vault-demo'

## CLEAN UP

Revoca primero el token almacenado en Kubernetes y después elimina el release, el namespace, la política y los temporales locales.

In [ ]:
%%bash
set -euo pipefail

if kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${MONITORING_NAMESPACE}" get secret vault-prometheus-token >/dev/null 2>&1; then
  metrics_token=$(kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${MONITORING_NAMESPACE}" get secret vault-prometheus-token -o jsonpath='{.data.token}' | base64 --decode)
  vault token revoke "${metrics_token}" >/dev/null || true
  unset metrics_token
fi
helm uninstall "${MONITORING_RELEASE}" --kube-context "${KUBE_CONTEXT:-$(kubectl config current-context)}" --namespace "${MONITORING_NAMESPACE}" --ignore-not-found
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} delete namespace "${MONITORING_NAMESPACE}" --ignore-not-found --wait=true
vault policy delete "${PROMETHEUS_POLICY}" >/dev/null || true
rm -rf "${WORKDIR}"
rm -f /tmp/vault-prometheus-port-forward.log /tmp/vault-grafana-port-forward.log /tmp/vault-metrics-sample.txt
echo 'Cleanup completado: monitoring, token y política eliminados.'